# Exercise 05b — OD features, ensembles, and diagnostics

This exercise uses the ten-zone Manhattan OD subset from Tutorial 05b. Data preparation is provided. You will change the feature structure, tune two ensemble models on validation data, compare all frozen models once on test, and diagnose errors by OD pair and time of day.

The raw flow target is retained throughout. Test data are reserved for the final comparison and diagnostic figures.

## Open this notebook

- **Colab.** Open the notebook, then run the setup cell below before any data-loading cell. It clones the public course repository and changes into `05 Transportation Applications I`, so relative data paths work.
- **On your own computer.** Open the notebook from its lesson folder so the data files sit beside it.


In [ ]:
from pathlib import Path
import os

def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

if in_colab():
    print("Running in Google Colab")
    REPO_URL = "https://github.com/hku-kejintao/AI-Code.git"
    LESSON_DIR = "05 Transportation Applications I"
    base = Path("/content")
    repo = base / "AI-Code"
    os.chdir(base)
    if not repo.exists():
        !git clone {REPO_URL}
    lesson_path = repo / LESSON_DIR
    if not lesson_path.exists():
        raise FileNotFoundError(f"Lesson folder not found: {lesson_path}")
    os.chdir(lesson_path)
else:
    print("Running on your own computer")

print("Current folder:", os.getcwd())


### Check the packages

The cell should print version numbers and the word `ready`. NumPy holds arrays, pandas holds tables, Matplotlib draws figures, and scikit-learn provides preprocessing, models, and metrics.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn

print("numpy", np.__version__)
print("pandas", pd.__version__)
print("matplotlib", plt.matplotlib.__version__)
print("scikit-learn", sklearn.__version__)
print("ready")


### Install, only if the check failed

Skip this cell when the check printed `ready`. After installation finishes, restart the kernel and run the cells above again.


In [ ]:
%pip install numpy pandas matplotlib scikit-learn


## Provided data preparation

In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor, ExtraTreesRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore")
np.random.seed(42)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False, "axes.titleweight": "bold"})

od = pd.read_csv("nyc_od_hour.csv.gz", parse_dates=["time"])
zones = pd.read_csv("nyc_zone_features.csv")
zone_names = zones.set_index("LocationID")["Zone"]
training_totals = od[od.time < "2024-02-10"].groupby("origin").flow.sum()
zone_ids = training_totals.nlargest(10).index.tolist()
d = od[od.origin.isin(zone_ids) & od.destination.isin(zone_ids)].sort_values(["origin", "destination", "time"]).copy()
for lag in [1, 2, 3, 24, 168]:
    d[f"lag_{lag}"] = d.groupby(["origin", "destination"])["flow"].shift(lag)
d["hour_sin"] = np.sin(2 * np.pi * d.time.dt.hour / 24)
d["hour_cos"] = np.cos(2 * np.pi * d.time.dt.hour / 24)
d["weekend"] = (d.time.dt.dayofweek >= 5).astype(int)
d = d.dropna().reset_index(drop=True)

feature_sets = {
    "recent": ["lag_1", "lag_2", "lag_3"],
    "+ daily/weekly": ["lag_1", "lag_2", "lag_3", "lag_24", "lag_168"],
    "+ calendar": ["lag_1", "lag_2", "lag_3", "lag_24", "lag_168", "hour_sin", "hour_cos", "weekend"],
}
full_features = feature_sets["+ calendar"]
train = d.time < "2024-02-10"
valid = d.time.between("2024-02-10", "2024-02-20", inclusive="left")
test = d.time >= "2024-02-20"
y = d.flow.to_numpy()

X_train = d.loc[train, full_features].to_numpy()
y_train = y[train]
X_valid = d.loc[valid, full_features].to_numpy()
y_valid = y[valid]
X_test = d.loc[test, full_features].to_numpy()
y_test = y[test]
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_valid:", X_valid.shape, "y_valid:", y_valid.shape)
print("X_test :", X_test.shape, "y_test :", y_test.shape)
print("Each row describes one OD pair at one target hour.")

def scores(name, observed, predicted):
    return {
        "model": name,
        "MAE": mean_absolute_error(observed, predicted),
        "RMSE": mean_squared_error(observed, predicted) ** .5,
        "R2": r2_score(observed, predicted),
    }

## Question 1 — Test temporal feature groups with Ridge

For each feature set, tune `alpha` in `[0.1, 1, 10]` using validation MAE. Freeze the best feature set and alpha together. Plot the validation result and explain whether daily and weekly OD histories add useful information beyond recent lags.

In [ ]:
# TODO: evaluate every feature-set × alpha combination on validation data.
# Store the selected pair as best_ridge_key = (feature_name, alpha).

## Question 2 — Tune two ensemble structures

Using the full feature set, tune Gradient Boosting over `learning_rate` in `[0.03, 0.07, 0.1]` and tune Extra Trees over `max_features` in `[0.5, 0.8, 1.0]`. Keep the remaining settings fixed. Select one configuration per family using validation MAE.

In [ ]:
# TODO: tune learning_rate for Gradient Boosting and max_features for Extra Trees.
# Store one selected key per family in best_ensemble_keys.

## Question 3 — Final frozen-model comparison

Refit the selected Ridge, Gradient Boosting, and Extra Trees models on train + validation. Evaluate each model on test once using MAE, RMSE, and $R^2$. Which conclusion is supported by the final table, and which conclusions would require more datasets?

In [ ]:
# TODO: refit each selected model on train + validation.
# TODO: evaluate all models once on test and plot MAE/RMSE.

## Question 4 — Diagnose OD-pair and time-of-day errors

For the test-best model, draw a named-zone heatmap of mean signed residuals (`observed − predicted`) for every OD pair and a curve of MAE by hour of day. List three OD pairs with the strongest systematic underprediction, and suggest information that could help explain those errors.

In [ ]:
# TODO: aggregate signed residual by (origin, destination) and absolute error by hour.
# TODO: draw the two diagnostics and list the three largest positive residuals.

## Question 5 — Extension: blend the two ensemble models

Choose a Gradient-Boosting weight from `[0, 0.25, 0.5, 0.75, 1]` using validation MAE. Freeze it, blend the final test predictions once, and compare it with both component models. State whether the blend adds useful diversity on this dataset.

In [ ]:
# TODO: select the blending weight on validation data and evaluate the frozen blend once on test.